<a href="https://colab.research.google.com/github/yanaguntikarmeesal/Book-Recommendation-System-project/blob/main/Book_Recommendation_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **📚 Google Colab — Book Recommendation System**

* Books, Ratings, and Users datasets and builds a collaborative-filtering recommendation system using cosine similarity

# Step 1 — Import required libraries

In [ ]:
# ============================================================
# STEP 1: IMPORT LIBRARIES
# ============================================================

# pandas is used for working with tables/data
import pandas as pd

# numpy is used for numerical operations
import numpy as np

print("Libraries imported successfully!")

# Step 2 — Upload the datasets

You need these 3 CSV files:

1. Books.csv
2. Ratings.csv
3. Users.csv

# Step 3 — Read the datasets

In [ ]:
# Read Books.csv
books = pd.read_csv("Books.csv")

In [ ]:
# Read Ratings.csv
ratings = pd.read_csv("Ratings.csv")

In [ ]:
# Read Users.csv
users = pd.read_csv("Users.csv")

# Step 4 — Look at the datasets

In [ ]:
print("BOOKS DATASET")
display(books.head())

In [ ]:
print("\nRATINGS DATASET")
display(ratings.head())

In [ ]:
print("\nUSERS DATASET")
display(users.head())

# 🔎 Step 5 — Check dataset shape

In [ ]:
print("Shape of Books dataset   :", books.shape)
print("Shape of Ratings dataset :", ratings.shape)
print("Shape of Users dataset   :", users.shape)

# 🔎 Step 6 — Check missing values

In [ ]:
print("Missing values in BOOKS:")
display(books.isnull().sum())

In [ ]:
print("\nMissing values in RATINGS:")
display(ratings.isnull().sum())

In [ ]:
print("\nMissing values in USERS:")
display(users.isnull().sum())

* In the source project, the Users dataset contains missing values in Age, while Ratings has no missing values.

# 🔎 Step 7 — Check duplicate rows

In [ ]:
print("Duplicate rows in Books   :", books.duplicated().sum())
print("Duplicate rows in Ratings :", ratings.duplicated().sum())
print("Duplicate rows in Users   :", users.duplicated().sum())

# 🔗 Step 8 — Merge Ratings and Books

We need book names along with ratings.

In [ ]:
# Merge using ISBN because ISBN identifies a book
ratings_with_name = ratings.merge(books, on="ISBN")

In [ ]:
# Display the first 5 rows
display(ratings_with_name.head())

In [ ]:
print("Shape after merging:", ratings_with_name.shape)

# STEP 9: COUNT NUMBER OF RATINGS FOR EACH BOOK

In [ ]:
# Group the data by Book-Title
# Then count how many ratings each book received

num_rating_df = (
    ratings_with_name
    .groupby("Book-Title")["Book-Rating"]
    .count()
    .reset_index()
)

In [ ]:
# Rename the column
num_rating_df.rename(
    columns={"Book-Rating": "num_ratings"},
    inplace=True
)

In [ ]:
# Display the result
display(num_rating_df.head())

# STEP 10: CALCULATE AVERAGE RATING

In [ ]:
# Make sure Book-Rating is numeric
ratings_with_name["Book-Rating"] = pd.to_numeric(
    ratings_with_name["Book-Rating"],
    errors="coerce"
)

In [ ]:
# Calculate average rating for every book
avg_rating_df = (
    ratings_with_name
    .groupby("Book-Title")["Book-Rating"]
    .mean()
    .reset_index()
)

In [ ]:
# Rename column
avg_rating_df.rename(
    columns={"Book-Rating": "avg_rating"},
    inplace=True
)

In [ ]:
display(avg_rating_df.head())

# 🔗 Step 11 — Combine number of ratings and average rating

In [ ]:
# ============================================================
# STEP 11: MERGE RATING COUNT AND AVERAGE RATING
# ============================================================

popular_df = num_rating_df.merge(
    avg_rating_df,
    on="Book-Title"
)

In [ ]:
display(popular_df.head())

# 🏆 Step 12 — Create Popularity-Based Recommendation System

The original project keeps books with at least 250 ratings and selects the top 50

In [ ]:
# ============================================================
# STEP 12: POPULARITY-BASED RECOMMENDER
# ============================================================

# Keep only books having at least 250 ratings
popular_df = popular_df[
    popular_df["num_ratings"] >= 250
]

In [ ]:
# Sort books by average rating
# Highest rating comes first
popular_df = popular_df.sort_values(
    "avg_rating",
    ascending=False
)

In [ ]:
# Keep only the top 50 books
popular_df = popular_df.head(50)

In [ ]:
display(popular_df)

# 📖 Step 13 — Add book details

In [ ]:
# ============================================================
# STEP 13: ADD BOOK DETAILS
# ============================================================

popular_df = popular_df.merge(
    books,
    on="Book-Title"
)

In [ ]:
# Remove duplicate books
popular_df = popular_df.drop_duplicates(
    "Book-Title"
)

In [ ]:
# Keep only useful columns
popular_df = popular_df[
    [
        "Book-Title",
        "Book-Author",
        "Image-URL-M",
        "num_ratings",
        "avg_rating"
    ]
]

In [ ]:
display(popular_df.head())

# 👥 Step 14 — Select experienced users

For collaborative filtering, the original project uses:

1. User must have more than 200 ratings
2. Book must have at least 50 ratings

In [ ]:
# ============================================================
# STEP 14: FIND EXPERIENCED USERS
# ============================================================

# Count how many books each user has rated
user_rating_count = (
    ratings_with_name
    .groupby("User-ID")["Book-Rating"]
    .count()
)

In [ ]:
# Keep users who have rated more than 200 books
experienced_users = user_rating_count[
    user_rating_count > 200
].index

In [ ]:
print("Number of experienced users:",
      len(experienced_users))

# 🔎 Step 15 — Filter ratings using experienced users

In [ ]:
# ============================================================
# STEP 15: FILTER RATINGS
# ============================================================

filtered_rating = ratings_with_name[
    ratings_with_name["User-ID"].isin(experienced_users)
]


In [ ]:
print("Shape of filtered ratings:",
      filtered_rating.shape)

In [ ]:
display(filtered_rating.head())

# 📚 Step 16 — Find famous books

In [ ]:
# ============================================================
# STEP 16: FIND BOOKS WITH AT LEAST 50 RATINGS
# ============================================================

# Count ratings for each book
book_rating_count = (
    filtered_rating
    .groupby("Book-Title")["Book-Rating"]
    .count()
)

In [ ]:
# Keep books having at least 50 ratings
famous_books = book_rating_count[
    book_rating_count >= 50
].index

In [ ]:
print("Number of famous books:",
      len(famous_books))

# 🔗 Step 17 — Create final ratings dataset

In [ ]:
final_ratings = filtered_rating[
    filtered_rating["Book-Title"].isin(famous_books)
]


In [ ]:
print("Shape of final ratings:",
      final_ratings.shape)

In [ ]:
display(final_ratings.head())

# 📊 Step 18 — Create Pivot Table

In [ ]:
pt = final_ratings.pivot_table(
    index="Book-Title",
    columns="User-ID",
    values="Book-Rating"
)

In [ ]:
# Display pivot table
display(pt.head())

In [ ]:
print("Pivot table shape:", pt.shape)

# 🧹 Step 19 — Replace missing values with 0

In [ ]:
# ============================================================
# STEP 19: HANDLE MISSING VALUES
# ============================================================

# A missing value means that the user
# has not rated that particular book.

# We replace missing values with 0.
pt.fillna(0, inplace=True)

In [ ]:
display(pt.head())

# 📐 Step 20 — Calculate Cosine Similarity

This is the main recommendation algorithm.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate similarity between all books
similarity_scores = cosine_similarity(pt)

print("Similarity matrix shape:",
      similarity_scores.shape)

# 🔍 Step 21 — Check similarity for one book

Let's use 1984.

In [ ]:
# Find the index of 1984
book_index = np.where(
    pt.index == "1984"
)[0][0]

In [ ]:
print("Index of 1984:", book_index)

In [ ]:
# Get similarity scores for 1984
similarity_for_book = similarity_scores[book_index]

In [ ]:
print("\nSimilarity scores:")
print(similarity_for_book[:10])

# 🥇 Step 22 — Sort similar books

In [ ]:
# ============================================================
# STEP 22: SORT BOOKS BY SIMILARITY
# ============================================================

similar_books = sorted(
    list(enumerate(similarity_for_book)),
    key=lambda x: x[1],
    reverse=True
)


In [ ]:
# Display first 10
print(similar_books[:10])

# 📚 Step 23 — Create Recommendation Function

In [ ]:
# ============================================================
# STEP 23: CREATE BOOK RECOMMENDATION FUNCTION
# ============================================================

def recommend(book_name):

    # Check whether the book exists
    if book_name not in pt.index:
        print("Sorry! This book is not available.")
        return

    # Find the index of the selected book
    index = np.where(
        pt.index == book_name
    )[0][0]

    # Get similarity scores for this book
    similar_items = sorted(
        list(enumerate(similarity_scores[index])),
        key=lambda x: x[1],
        reverse=True
    )[1:6]

    # Print the top 5 similar books
    print(f"\nBooks similar to: {book_name}\n")

    for item in similar_items:

        # Get book name using index
        recommended_book = pt.index[item[0]]

        # Similarity score
        score = item[1]

        print(
            f"{recommended_book} "
            f"(Similarity: {score:.3f})"
        )

# 🧪 Step 24 — Test the Recommendation System

In [ ]:
# ============================================================
# STEP 24: TEST THE MODEL
# ============================================================

recommend("1984")

* The source notebook demonstrates recommendations for 1984, including titles such as The Handmaid's Tale, Fahrenheit 451, and The Catcher in the Rye

# Try another book:

In [ ]:
recommend("Zoya")

# Or:

In [ ]:
recommend("Wuthering Heights")

# 🔍 Step 25 — Let the user enter a book

This makes the Colab notebook interactive.

In [ ]:
# ============================================================
# STEP 25: INTERACTIVE RECOMMENDATION
# ============================================================

book_name = input(
    "Enter a book name: "
)

recommend(book_name)

# Example:

Enter a book name: 1984

# 💾 Step 26 — Save the Popular Books Model

The original project saves Popular_Books_Model_df using pickle.

In [ ]:
# ============================================================
# STEP 26: SAVE POPULAR BOOK DATA
# ============================================================

import pickle

# Save popular books
with open(" popular.pkl", "wb") as file:
    pickle.dump(popular, file)

print(" popular.pkl created successfully!")

# 💾 Step 27 — Save Recommendation Data

For a Streamlit application, it is useful to save the pivot table and similarity matrix too.

In [ ]:
# ============================================================
# STEP 27: SAVE RECOMMENDATION MODEL
# ============================================================

# Save pivot table
with open("pt.pkl", "wb") as file:
    pickle.dump(pt, file)

# Save similarity scores
with open("similarity_scores.pkl", "wb") as file:
    pickle.dump(similarity_scores, file)

print("Recommendation files saved successfully!")

# 📥 Step 28 — Download the files

In [ ]:
# ============================================================
# STEP 28: DOWNLOAD MODEL FILES
# ============================================================

from google.colab import files

files.download("Popular_Books_Model_df.pkl")

In [ ]:
files.download("pt.pkl")

In [ ]:
files.download("similarity_scores.pkl")

# 🧠 Complete Project Flow

The entire project works like this:

             BOOK RECOMMENDATION SYSTEM
                       │
                       ▼
              Load CSV datasets
                       │
          ┌────────────┼────────────┐
          ▼            ▼            ▼
       Books        Ratings       Users
          │            │            │
          └────────────┼────────────┘
                       ▼
                Merge datasets
                       │
                       ▼
             Popularity Analysis
                       │
                       ▼
              Top Popular Books
                       │
                       ▼
              Filter Users
           (> 200 ratings)
                       │
                       ▼
              Filter Books
           (>= 50 ratings)
                       │
                       ▼
                Pivot Table
            Books × Users
                       │
                       ▼
             Cosine Similarity
                       │
                       ▼
             Similarity Matrix
                       │
                       ▼
              Recommendation
                       │
                       ▼
             Top 5 Similar Books

# 🎯 What you are learning from this project

| Step              | Concept                          |
| ----------------- | -------------------------------- |
| `read_csv()`      | Loading datasets                 |
| `head()`          | Exploring data                   |
| `shape`           | Understanding dataset size       |
| `isnull()`        | Missing-value analysis           |
| `duplicated()`    | Duplicate detection              |
| `merge()`         | Combining datasets               |
| `groupby()`       | Aggregation                      |
| `mean()`          | Average rating                   |
| `pivot_table()`   | Creating user-book matrix        |
| Filtering         | Selecting useful users/books     |
| Cosine similarity | Measuring book similarity        |
| Pickle            | Saving ML/recommendation objects |
